# 🎬 Geração de Vídeo Cinematográfico a partir de Imagens

In [ ]:
!pip install diffusers accelerate einops transformers ffmpeg-python gradio --quiet

In [ ]:
import os
import torch
import ffmpeg
from PIL import Image
import gradio as gr
from diffusers import StableVideoDiffusionPipeline
from google.colab import files

os.makedirs("inputs", exist_ok=True)
os.makedirs("outputs", exist_ok=True)


In [ ]:
def generate_video(image_path, style_prompt, duration=2, fps=14):
    try:
        print(f"Iniciando geração para: {image_path} | Estilo: {style_prompt}")
        
        model_id = "stabilityai/stable-video-diffusion-img2vid-xt"
        pipe = StableVideoDiffusionPipeline.from_pretrained(
            model_id, torch_dtype=torch.float16, variant="fp16"
        )
        pipe = pipe.to("cuda")

        image = Image.open(image_path).convert("RGB")
        image = image.resize((576, 1024))

        output = pipe(prompt=style_prompt, image=image, decode_chunk_size=8, num_frames=25)

        if hasattr(output, "frames"):
            frames = output.frames
        elif "frames" in output:
            frames = output["frames"]
        else:
            raise ValueError("Frames não encontrados na saída do modelo.")

        frame_dir = "outputs/temp_frames"
        os.makedirs(frame_dir, exist_ok=True)

        for idx, frame in enumerate(frames):
            frame.save(f"{frame_dir}/frame_{idx:03d}.png")

        output_filename = os.path.basename(image_path).split(".")[0] + ".mp4"
        output_path = os.path.join("outputs", output_filename)

        ffmpeg.input(f"{frame_dir}/frame_%03d.png", framerate=fps)\
              .output(output_path, vcodec='libx264', pix_fmt='yuv420p')\
              .run(overwrite_output=True)

        print("Vídeo gerado com sucesso:", output_path)
        return output_path

    except Exception as e:
        print("❌ Erro ao gerar o vídeo:", str(e))
        return None


In [ ]:
def video_app(img, style, duration, fps):
    result = generate_video(img, style, duration, fps)
    if result is None:
        raise gr.Error("Erro durante a geração do vídeo. Verifique o console do notebook.")
    return result


In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## 🎥 Geração de Vídeos Cinematográficos a partir de Imagens")
    with gr.Row():
        img_input = gr.Image(label="Imagem de Entrada", type="filepath")
        style_input = gr.Textbox(label="Prompt de Estilo (ex: 'anime watercolor, cinematic')", value="cinematic")
    with gr.Row():
        duration_input = gr.Slider(label="Duração por Clipe (s)", minimum=1, maximum=10, step=0.5, value=2)
        fps_input = gr.Slider(label="FPS", minimum=6, maximum=30, step=1, value=14)
    generate_btn = gr.Button("Gerar Vídeo")
    video_output = gr.Video(label="Resultado")

    generate_btn.click(fn=video_app, inputs=[img_input, style_input, duration_input, fps_input], outputs=video_output)

demo.launch(share=True, debug=True)


In [ ]:
# Após geração, você pode baixar manualmente o vídeo aqui ou usar:
# from google.colab import files
# files.download("outputs/SEU_VIDEO.mp4")